[Guide Index](README.md)

# 02d The Oxigraph-backed graph (`StarLayerGraph` / `StarLayerDataset`)

`StarLayerGraph` and `StarLayerDataset` wrap a `pyoxigraph.Store`. See `docs/working/oxgraph-wrapper.md` for the design. This guide covers the boundary between the wrapper and Oxigraph: what the wrapper does for you, what you reach through `.store`, and where behaviour differs from calling Oxigraph directly.

pyoxigraph's own classes cannot be subclassed, so the wrapper holds the store rather than extending it. Anything not listed here is documented by [pyoxigraph](https://pyoxigraph.readthedocs.io/).

## Basics

In [1]:
from rdflib import URIRef, Literal
from starlayer.graph.oxgraph import StarLayerGraph, StarLayerDataset, as_store

g = StarLayerGraph(prefixes={"ex": "http://example.org/"})
g.parse(data='''@prefix ex: <http://example.org/> .
ex:a ex:knows ex:b ; ex:age 4 .
ex:b ex:knows ex:c .''', format="turtle")
print(len(g), "triples")
print(g.serialize(format="turtle"))

3 triples
@prefix ex: <http://example.org/> .
ex:b ex:knows ex:c .
ex:a ex:age 4 ;
	ex:knows ex:b .



## Querying

`query()` sends the SPARQL text straight to Oxigraph. `initNs` becomes Oxigraph's per-call `prefixes=`, and `initBindings` becomes `substitutions=`, which Oxigraph only applies to a variable that the query projects. The result is a `Result` wrapper that iterates rows.

In [2]:
EX = "http://example.org/"
result = g.query("SELECT ?s ?o WHERE { ?s ex:knows ?o }", initNs={"ex": EX})
for row in result:
    print(row["s"], "knows", row["o"])

bound = g.query("SELECT ?s ?o WHERE { ?s ex:knows ?o }", initNs={"ex": EX}, initBindings={"s": URIRef(EX + "a")})
print([str(r["o"]) for r in bound])

<http://example.org/b> knows <http://example.org/c>
<http://example.org/a> knows <http://example.org/b>
['<http://example.org/b>']


## The store boundary: `.store` and `as_store()`

`g.store` is the underlying `pyoxigraph.Store`; `as_store(x)` returns it for a wrapper and passes a `Store` through unchanged. Seven `Store` methods are also wrapped (`add`, `clear`, `extend`, `query`, `remove`, `remove_graph`, `update`). The rest are reached through `.store`. The wrapper does **not** forward unknown attributes, so `g.flush()` is an `AttributeError`.

In [3]:
import pyoxigraph as ox

store = g.store
print(type(store).__name__, as_store(g) is store)
wrapped = {"add", "clear", "extend", "query", "remove", "remove_graph", "update"}
public = {m for m in dir(ox.Store) if not m.startswith("_")}
print("only through .store:", sorted(public - wrapped))

Store True
only through .store: ['add_graph', 'backup', 'bulk_extend', 'bulk_load', 'clear_graph', 'contains_named_graph', 'dump', 'flush', 'load', 'named_graphs', 'optimize', 'quads_for_pattern', 'read_only']


In [4]:
try:
    g.flush()
except AttributeError as error:
    print("AttributeError:", error)
store.flush()  # persistence operations go through the store

AttributeError: 'StarLayerGraph' object has no attribute 'flush'


## Persistence

A path opens an on-disk (RocksDB) store. Only one process may write to it at a time, and there are no transactions beyond the atomicity of a single `update`.

In [5]:
import tempfile

path = tempfile.mkdtemp()
disk = StarLayerGraph(path=path)
disk.add((URIRef(EX + "s"), URIRef(EX + "p"), Literal("kept")))
disk.store.flush()
print(len(disk), "triple on disk")

1 triple on disk


## Set operations

The operators `+`, `-`, `&` and `^` work on wrappers; `starlayer.graph.setops` offers the same operations as functions that also accept bare `pyoxigraph` stores. Blank nodes are compared by label, not by isomorphism.

In [6]:
from starlayer.graph import setops

other = StarLayerGraph()
other.add((URIRef(EX + "a"), URIRef(EX + "knows"), URIRef(EX + "b")))
print("difference:", len(g - other), " intersection:", len(g & other), " union:", len(g + other), " symmetric:", len(g ^ other))
print("on bare stores:", len(setops.difference(g.store, other.store)))

difference: 2  intersection: 1  union: 3  symmetric: 2
on bare stores: 2


## Where Oxigraph differs from rdflib

**The store rewrites some literals on insert** (issue #190). A lexical form such as `"04"^^xsd:integer` is stored as `4`; the original text is not recoverable from the store. Ill-formed values and strings are kept as written.

In [7]:
h = StarLayerGraph()
h.parse(data='<http://e/a> <http://e/age> "04"^^<http://www.w3.org/2001/XMLSchema#integer> .', format="turtle")
print([str(o) for o in h.objects()])

['"4"^^<http://www.w3.org/2001/XMLSchema#integer>']


**RDF 1.1 output.** The wrapper stores RDF 1.2 terms, including triple terms. To hand a graph to an RDF 1.1 consumer, serialise with `rdf_version="1.1"`; triple terms become the "basic encoding" (`rdf:PropositionForm` and its subject, predicate and object properties).

In [8]:
t = StarLayerGraph()
t.parse(data='''PREFIX ex: <http://example.org/>
ex:a ex:says <<( ex:b ex:knows ex:c )>> .''', format="turtle")
print(t.serialize(format="turtle", rdf_version="1.1"))

@prefix ex: <http://example.org/> .
ex:a ex:says _:ff90998f4be0071684a95682e8d0b395 .
_:ff90998f4be0071684a95682e8d0b395 a <http://www.w3.org/1999/02/22-rdf-syntax-ns#PropositionForm> ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormSubject> ex:b ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormPredicate> ex:knows ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormObject> ex:c .



## Datasets

`StarLayerDataset` holds named graphs in one store.

In [9]:
d = StarLayerDataset()
d.parse(data='<http://e/s> <http://e/p> <http://e/o> <http://e/g> .', format="nquads")
print(len(d), "quad;", "store type:", type(d.store).__name__)

1 quad; store type: Store
